# prepare_new_external_yaml.ipynb — Preparing and Validating a New External YAML for IDTrack

*Last updated:* 2025-09-02 12:53 UTC

This notebook is the **canonical, step‑by‑step guide** for creating and validating a new *external YAML* configuration
to integrate a **new organism** (or update an existing one) and select **external databases** to include in the
cross‑database identifier graph built by **IDTrack**.

Why this matters:

- IDTrack builds a time‑aware graph of identifiers (Ensembl releases, genome assemblies, and external databases) to
  support **robust cross‑database mapping** across versions and data sources.
- The *external YAML* is a small, human‑editable file that **declares which external databases** (HGNC, RefSeq, UniProt,
  RFAM, …) should be included for a given organism, Ensembl release range, and genome assembly.
- A clean, correct YAML helps ensure graph construction is reproducible and the path‑finding step remains **fast,
  interpretable, and stable**.

> This tutorial expands the concise developer notebook you may have seen into a full, publication‑quality guide. It
> covers prerequisites, the YAML format, how to generate a starter file, what to edit, and how to validate & test it.

## 1. Pre‑requisites

- **Software**: `python (>=3.9)`, `idtrack` (installed), plus common helpers we use in a few cells: `pandas`, `pyyaml`.
- **Network access**: for first‑time metadata downloads and cache fills.
- **A working folder** (the *local repository*) where IDTrack will cache data and where your YAML lives.
- **Version control**: create a short‑lived Git branch for your YAML changes. Commit both the YAML and a small text file
  (or README) that records *what you changed and why* (source, date, releases/assemblies covered).

> **Tip:** if you contribute a new default YAML to the IDTrack repository, work in a feature branch and open a Pull
> Request titled _“Add externals config for <organism>”_. Include notes about **assemblies**, **release ranges**, and
> **chosen databases**.

In [ ]:
# Quick environment/report cell: safe to run even if idtrack isn't installed.
import sys, os, importlib, textwrap

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

try:
    import idtrack
    print("idtrack version:", getattr(idtrack, "__version__", "unknown"))
    IDTRACK_AVAILABLE = True
except Exception as e:
    print("idtrack import failed ->", repr(e))
    IDTRACK_AVAILABLE = False

# Optional helpers used in a few cells
try:
    import pandas as pd  # noqa: F401
    import yaml  # PyYAML  # noqa: F401
    print("PyYAML & pandas available")
except Exception as e:
    print("Warning: pandas / pyyaml not available ->", repr(e))

### Directory layout & where YAML files live

There are two relevant locations:

1. **Your local repository** (arbitrary path you pick). IDTrack caches downloads here and will look for your edited
   YAML named `"<organism>_externals_modified.yml"`.
2. **The package default config** inside the installed `idtrack` package under `default_config/`. This holds read‑only
   fallbacks used for tests or as a starting point.

You will normally **create your YAML in (1)**. If you plan to contribute it to the package, you will later move it (in a
Pull Request) into location **(2)**.

In [ ]:
# Discover sensible defaults for paths.
from pathlib import Path

# 1) Choose or create a local repository path
LOCAL_REPOSITORY = Path.home() / "idtrack_local"
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

print("Local repository will be:", LOCAL_REPOSITORY)

# 2) If idtrack is available, show where the package lives and its default_config/
if 'idtrack' in sys.modules:
    pkg_root = Path(idtrack.__file__).resolve().parent
    default_cfg = pkg_root / "default_config"
    print("idtrack package root :", pkg_root)
    print("default_config path  :", default_cfg, "(exists:", default_cfg.exists(), ")")
else:
    print("idtrack not imported; skip package path discovery.")

## 2. Overview of the external YAML format

IDTrack generates a **template** YAML that you edit. The structure is nested as:

```
<organism>:
  <form>:                          # e.g. 'gene'
    <External DB name A>:
      Assembly:
        <assembly_code>:           # integer like 38 (GRCh38), 37 (GRCh37), 102 (GRCm39), ...
          Ensembl release: "80,81,82,...,114"   # comma‑separated releases in which this pairing exists
          Include: false           # <-- YOU toggle this to true for databases you want
          Database Index: 0        # currently informational (reserved)
          Potential Synonymous: [] # currently informational (reserved)
    <External DB name B>:
      Assembly:
        <assembly_code>:
          Ensembl release: "..."
          Include: false
          Database Index: 0
          Potential Synonymous: []
```

**Key fields**

- **`Include` (bool)** – set to `true` for databases you want to include in the graph.
- **`Ensembl release` (str)** – comma‑separated releases where that database↔form is available for the assembly.
- **`Assembly` (map[int → obj])** – assemblies (by numeric code) for which this database/form pairing exists.
- **`Database Index`**, **`Potential Synonymous`** – informational placeholders; kept for future tooling.
- **Top‑level keys `organism` and `form`** are implicit from the nesting.

> **Note:** You do **not** need to add URLs or GTF/GFF paths here. This file governs *selection* of external databases
> (HGNC, RefSeq, UniProt, …) **as defined by Ensembl** for the organism, form, assembly, and release range.

## 3. Prepare a new external YAML entry (from scratch)

We'll go end‑to‑end:
1. Normalise your organism name and discover the **latest supported Ensembl release**.
2. Create a `DatabaseManager` snapshot **bounded by that release** (reproducible).
3. Download/assemble the **external database metadata table** across assemblies/releases.
4. Generate a **template YAML** in your local repository.
5. **Edit** the template (toggle `Include: true` for chosen databases) and **save as** `*_externals_modified.yml`.
6. Validate it.

In [ ]:
# 1) Create the high-level API facade
from pathlib import Path

if not IDTRACK_AVAILABLE:
    raise RuntimeError("Please install `idtrack` in this environment before running the tutorial cells.")

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))

# 2) Resolve organism name & latest supported Ensembl release
#    You can pass a synonym / free text like "human", "H. sapiens", "9606", etc.
tentative = "homo sapiens"
organism_name, latest_release = api.resolve_organism(tentative)
print("Resolved organism  :", organism_name)
print("Latest Ensembl rel.:", latest_release)

# You can also pin an older 'snapshot_release' for reproducibility, if desired:
SNAPSHOT_RELEASE = latest_release  # change to a lower integer to reproduce historical snapshots

In [ ]:
# 3) Create a DatabaseManager for the organism + snapshot release
dm = api.get_database_manager(organism_name=organism_name, snapshot_release=SNAPSHOT_RELEASE)
dm

In [ ]:
# 4) Retrieve external-database metadata (slow on first run; cached thereafter)
#    Set 'just_download=True' if you only want to populate caches without keeping the big dataframe in memory.
df = dm.create_database_content(just_download=False)

print("Rows:", len(df))
print("Columns:", list(df.columns))
print(df.head(3))

In [ ]:
# 5) Generate the TEMPLATE YAML
template_path = dm.external_inst.file_name_template_yaml()
dm.external_inst.create_template_yaml(df)

print("Template written to:", template_path)
print("\nPreview (first ~80 lines):\n")
try:
    with open(template_path, "r", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i > 80: break
            print(line.rstrip())
except FileNotFoundError:
    print("Template not found. Check LOCAL_REPOSITORY and rerun the previous cell.")

### How to edit the template

1. Open the file shown above (ends with `_externals_template.yml`).
2. For each **external database** you want to include, locate the **assembly** you are targeting and set `Include: true`.
3. Keep the `Ensembl release` list as generated—this expresses **in which releases** that pairing exists.
4. **Save as a new file** with the suffix `*_externals_modified.yml` (same folder). Example:

- Template   → `homo_sapiens_externals_template.yml`
- **Modified → `homo_sapiens_externals_modified.yml`**  ← IDTrack will look for this first.

> **Practical advice:** prefer **high‑quality, non‑redundant** databases (e.g. HGNC, Entrez/NCBI Gene, UniProt, RefSeq).
> Enabling too many near‑synonymous sources makes the path‑finding stage noisier (more branches), which may slow down or
> complicate interpretation. Choose **few, strong** sources over **many, overlapping** ones.

In [ ]:
# (Optional) Programmatic edit example: toggle Include: true for a short allowlist
# Safe to run multiple times; will only change entries that exist.
import yaml
from pathlib import Path

modified_path = Path(str(template_path).replace("_template.yml", "_modified.yml"))

with open(template_path, "r", encoding="utf-8") as fh:
    y = yaml.safe_load(fh)

ALLOWLIST = {
    "HGNC Symbol",
    "EntrezGene",
    "UniProtKB Gene Name",
    "RefSeq_mRNA",  # Example names; your template may include slightly different labels
}
changed = 0

the_form = list(y.get(organism_name, {}).keys())[0] if y.get(organism_name) else None
if the_form is None:
    raise RuntimeError("Unexpected template structure—no form key found under organism. Inspect the template manually.")

for db_name, db_block in y[organism_name][the_form].items():
    if db_name in ALLOWLIST:
        asm_map = db_block.get("Assembly", {})
        for asm, attrs in asm_map.items():
            if attrs.get("Include") is False:
                attrs["Include"] = True
                changed += 1

print(f"Toggled Include: true in {changed} (db x assembly) entries.")
with open(modified_path, "w", encoding="utf-8") as fh:
    yaml.safe_dump(y, fh, sort_keys=False, allow_unicode=True)
print("Saved:", modified_path)

## 4. Validation and sanity checks

We recommend the following validation sequence.

1. **Schema/keys check:** load the `*_externals_modified.yml` and ensure required sections exist.
2. **Release coverage check:** confirm your **snapshot release** is present among the `Ensembl release` lists.
3. **Preview selections:** list the enabled **databases** and **assemblies** that IDTrack will use.
4. **(Optional) Dry‑run graph build:** construct a **test** track to ensure the YAML is honored.

> The `ExternalDatabases.load_modified_yaml()` helper **first looks for your modified file** in the local repository.
> If absent, it falls back to the package defaults and warns you.

In [ ]:
# 1 & 2) Load + validate against the snapshot release
validated = dm.external_inst.load_modified_yaml()  # raises if snapshot_release isn't covered
print("YAML loaded OK. Top-level keys:", list(validated.keys())[:5])

In [ ]:
# 3) Preview selections that will be used for this (organism, form, snapshot_release, assembly)
enabled_dbs = dm.external_inst.give_list_for_case("db")
assemblies = dm.external_inst.give_list_for_case("assembly")
print("Enabled DBs for assembly", dm.genome_assembly, ":", enabled_dbs)
print("Assemblies covered by YAML:", assemblies)

In [ ]:
# 4) Optional: Build a small test track (no heavy caches) to confirm the YAML is respected
#    This may download data on first run; caches live under LOCAL_REPOSITORY.
TEST_ONLY = True
api.build_graph(organism_name=organism_name, snapshot_release=SNAPSHOT_RELEASE, return_test=TEST_ONLY, calculate_caches=False)

print("Track object type:", type(api.track).__name__)
print("External DBs considered by the graph for this assembly:", api.track.graph.external_databases_for_assembly)

## 5. Submitting your YAML (for local use or for contribution)

- **For local workflows**: keep `*_externals_modified.yml` in your **local repository**. Share it with collaborators
  alongside a short README describing your choices.
- **For contribution to IDTrack**:
  1. Fork/branch the repository.
  2. Add your file under the package's `default_config/` directory (follow existing naming convention).
  3. Update or add a small test if appropriate.
  4. Run the developer checks (formatting, typing, docs build, tests).
  5. Open a Pull Request titled _“Add externals config for `<organism>` (assembly `<code>`)”_.

**Recommended checks**

- `pre-commit run -a`
- `pytest -q`
- Type checks (if enabled locally): `mypy idtrack`
- Build docs if you've updated documentation/examples.

## 6. Example: human (Homo sapiens) with GRCh38

Below is a minimal end‑to‑end flow. It mirrors the steps above but is ready to execute once `idtrack` is installed.
Feel free to adapt the allowlist to your curation policy.

In [ ]:
# Resolve organism & release, create manager, fetch metadata, create/edit/validate YAML, then preview.
api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
organism_name, latest_release = api.resolve_organism("homo sapiens")
dm = api.get_database_manager(organism_name=organism_name, snapshot_release=latest_release)

df = dm.create_database_content(just_download=False)
dm.external_inst.create_template_yaml(df)

# Programmatic Include toggles (adjust as desired); then validate & preview
template_path = dm.external_inst.file_name_template_yaml()
modified_path = str(template_path).replace("_template.yml", "_modified.yml")

import yaml, pathlib
y = yaml.safe_load(open(template_path, "r", encoding="utf-8"))
the_form = list(y[organism_name].keys())[0]
for db in ["HGNC Symbol", "EntrezGene", "UniProtKB Gene Name", "RefSeq_mRNA"]:
    if db in y[organism_name][the_form]:
        for asm, attrs in y[organism_name][the_form][db]["Assembly"].items():
            attrs["Include"] = True
yaml.safe_dump(y, open(modified_path, "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)

# Validate + preview
_ = dm.external_inst.load_modified_yaml()
print("Enabled DBs:", dm.external_inst.give_list_for_case("db"))

## 7. Common pitfalls & troubleshooting

- **YAML indentation**: YAML is whitespace‑sensitive. Use **two spaces per level** and avoid tabs.
- **Wrong filename**: IDTrack looks for `*_externals_modified.yml`. Make sure you saved with this suffix (same directory
  as the template).
- **Missing snapshot release**: If your chosen `SNAPSHOT_RELEASE` is **not** present in the `Ensembl release` lists for
  the target assembly, validation raises an error. Either **lower your snapshot release** or **extend the YAML**.
- **Too many external databases enabled**: Selecting many near‑synonymous databases (e.g., both SWISSPROT and TREMBL plus
  multiple RefSeq types) **inflates the graph** and can slow path‑finding or complicate interpretation. Prefer a small
  set of **high‑quality, non‑redundant** sources.
- **Assembly mismatch**: If you intend to work with multiple assemblies (e.g., GRCh37 and GRCh38), review each assembly
  block in the YAML and set `Include: true` accordingly.
- **Environment differences**: Caches are bound to your `LOCAL_REPOSITORY`. If collaborators run into issues, confirm
  they use the same snapshot release and have read access to the YAML and caches.

## 8. Appendix

- **Where the fields come from:** the `Ensembl release` lists and the available external database names are derived from
  Ensembl's `external_db` metadata for the chosen organism/form/assembly. IDTrack aggregates this into the template.
- **Choosing wisely:** aim for databases that (a) offer high‑quality curation, (b) connect meaningfully to your use case,
  and (c) **do not cause unnecessary branching** in the graph. A lean set of strong sources tends to produce cleaner,
  more stable mappings.
- **Advanced usage:** you can maintain **multiple modified YAML files** (e.g., one per assembly or project) by changing
  the filename convention in your own tooling. IDTrack itself expects the suffix `_externals_modified.yml` in your local
  repository, and will fall back to packaged defaults if not found.
- **Automation:** If you curate many organisms, wrap the generation + programmatic toggling into a small script with a
  simple allowlist per organism, and commit both the script and resulting YAML to your repo.